In [12]:
# Cell 1 — Import libraries and load the raw data
import pandas as pd
import numpy as np

RAW_PATH = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(RAW_PATH)

print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded 7,043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
# Cell 2 — Correct TotalCharges from text to a real number
# It came in as str because some rows have a blank string instead of a
# number. Strip whitespace first, then convert - anything that still
# can't be read as a number becomes NaN instead of crashing the notebook.
df["TotalCharges"] = df["TotalCharges"].astype(str).str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"TotalCharges dtype is now: {df['TotalCharges'].dtype}")
print(f"Values that failed to convert (now NaN): {df['TotalCharges'].isna().sum()}")

TotalCharges dtype is now: float64
Values that failed to convert (now NaN): 11


In [14]:
# Cell 3 — Fix the missing TotalCharges values
# Check who's missing before filling anything in
missing_rows = df[df["TotalCharges"].isna()]
print("Rows with missing TotalCharges:")
print(missing_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

# All of them have tenure == 0 - no billing period completed yet, so the
# true value is 0, not an unknown number to estimate.
print("\ntenure values for these rows:", missing_rows["tenure"].unique())

df["TotalCharges"] = df["TotalCharges"].fillna(0)

print(f"\nRemaining missing values in TotalCharges: {df['TotalCharges'].isna().sum()}")

Rows with missing TotalCharges:
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55           NaN
753   3115-CZMZD       0           20.25           NaN
936   5709-LVOEQ       0           80.85           NaN
1082  4367-NUYAO       0           25.75           NaN
1340  1371-DWPAZ       0           56.05           NaN
3331  7644-OMVMY       0           19.85           NaN
3826  3213-VVOLG       0           25.35           NaN
4380  2520-SGTTA       0           20.00           NaN
5218  2923-ARZLG       0           19.70           NaN
6670  4075-WKNIU       0           73.35           NaN
6754  2775-SEFEE       0           61.90           NaN

tenure values for these rows: [0]

Remaining missing values in TotalCharges: 0


In [15]:
# Cell 4 — Test cell: confirm the fixes actually worked, all in one place
assert df["TotalCharges"].dtype == "float64", "TotalCharges is not float64"
assert df["TotalCharges"].isna().sum() == 0, "TotalCharges still has missing values"
assert (df.loc[df["tenure"] == 0, "TotalCharges"] == 0).all(), "Not every tenure==0 row has TotalCharges == 0"
assert df.shape[0] == 7043, "Row count changed - a row was dropped or added unexpectedly"

print("All checks passed.")
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Missing values: {df['TotalCharges'].isna().sum()}")
print(f"Rows: {df.shape[0]:,}")

All checks passed.
TotalCharges dtype: float64
Missing values: 0
Rows: 7,043


In [ ]:
# Cell 5 — Export the cleaned data as a CSV in the current working directory
df.to_csv("telco_churn_cleaned.csv", index=False)

print("Saved telco_churn_cleaned.csv") # added to re commit

Saved telco_churn_cleaned.csv
